# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and process the FAIR^2 Mass Spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all data entities by their `@id`.

### Dataset Source
The Croissant schema describing the dataset can be found at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset metadata and preview basic descriptive information. All dataset interaction will use entity `@id` fields where applicable.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not as dict)
meta = dataset.metadata

print(f"Dataset name: {meta.name}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"Description: {meta.description}")

## 2. Data Overview
Let's review the available record sets (tables) in the dataset, along with their `@id`s and contained fields. Croissant datasets may contain multiple record sets—each representing data at different levels or tables.

In [ ]:
# List record sets (`cr:RecordSet`) and their @id
record_set_dict = {rs['@id']: rs for rs in getattr(meta, 'record_set', [])}

if not record_set_dict:
    print("No record sets found in metadata. The dataset may define record sets in the schema only. Let's try loading record sets from the dataset.records().")

# Try to enumerate record set ids via mlcroissant directly
from mlcroissant.types import RecordSet

# mlcroissant exposes record_set ids in dataset._metadata.record_set
if not record_set_dict:
    if hasattr(dataset.metadata, 'record_set') and dataset.metadata.record_set:
        print("Record sets as listed in metadata:")
        for rs in dataset.metadata.record_set:
            print(rs['@id'], '-', rs.get('name', ''))
    else:
        # Fallback: check schema structure
        print("Record set info not present in metadata. Attempting to list available data...\n")

# Find available record set ids using dataset.schema
record_set_ids = dataset.schema.record_sets
for rs_id, rs in record_set_ids.items():
    print(f"RecordSet @id: {rs_id}\n  Name: {rs.name}\n  Description: {getattr(rs, 'description', '-')}")
    print("  Field @ids:")
    for f in rs.fields.values():
        print(f"    - {f.id} (name: {f.name})")
    print('-' * 40)

## 3. Data Extraction
Now we'll load data from a specific record set into a pandas DataFrame. Record sets, fields, and columns are referenced **by their `@id`**. For demonstration, let's select one record set (typically the main table in the dataset).

In [ ]:
# List available record set ids and pick the main one for import
record_set_ids = list(dataset.schema.record_sets.keys())
print("Record set ids:", record_set_ids)

# Choose the primary table (usually the protein data table)
primary_record_set_id = record_set_ids[0]  # Replace 0 if another is main
print(f"Selected RecordSet @id: {primary_record_set_id}")

# Load records for the selected record set
records = list(dataset.records(record_set=primary_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records.")
print("Field @ids in DataFrame (as columns):")
print(df.columns.tolist())
# Preview first rows
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic filtering, normalization, and grouping operations. All columns/fields are referenced directly by their `@id` (as exposed in the DataFrame above).

In [ ]:
# Choose a numeric field for analysis by inspecting field @ids and their type
rs = dataset.schema.record_sets[primary_record_set_id]
numeric_fields = [
    f.id for f in rs.fields.values() if getattr(f, 'data_type', None) in ['schema:Float', 'schema:Number', 'schema:Integer']
        and f.id in df.columns
]
print(f"Numeric field @ids: {numeric_fields}")

# For demo: select the first numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError('No numeric fields detected.')

# Filtering: keep only records with numeric field above a threshold
threshold = df[numeric_field_id].dropna().mean()  # Use mean as example threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization: mean-std for filtered values
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized field {numeric_field_id}:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: pick a categorical/group field (e.g., protein type or sample), by string data type
cat_fields = [f.id for f in rs.fields.values() if getattr(f, 'data_type', None) == 'schema:Text' and f.id in df.columns]
print(f"Candidate group fields (@id): {cat_fields}")
group_field_id = cat_fields[0] if cat_fields else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('group_mean')
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and the result of grouping by the chosen categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of original numeric field
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group, if applicable
if group_field_id:
    plt.figure(figsize=(12,4))
    order = filtered_df[group_field_id].value_counts().index[:20]  # Top 20 groups
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
We loaded and explored a mass spectrometry dataset using `mlcroissant`, referencing all entities by their `@id`. We previewed table structures, extracted record data, performed column-level EDA (filtering and normalization), and visualized the distribution of a key quantitative field. This workflow can be extended to deeper analyses and integration with ML pipelines, always relying on semantic, explicit references to Croissant data structures.